In [1]:
corpus = '''
the cat sat on the mat
the dog sat on the log
the cat ate the rat
the rat ran from the cat
the dog ran after the rat
the bird flew over the tree
the squirrel climbed the tree
the rabbit hopped in the field
the fox chased the rabbit
the owl perched on the branch
the mouse scurried under the bush
the frog leaped into the pond
the duck swam in the lake
the goose honked loudly
the hawk soared in the sky
the bear lumbered through the forest
the deer grazed in the meadow
the wolf howled at the moon
the snake slithered in the grass
the lizard basked in the sun
the turtle rested on the rock
the butterfly fluttered from flower to flower
the bee buzzed around the hive
the ant marched in a line
the grasshopper chirped in the tall grass
the dragonfly darted over the water
the moth fluttered near the light
the spider spun its web
the worm crawled in the soil
the snail left a slimy trail
'''

In [2]:
import torch
import numpy
import math
import os
torch.manual_seed(35)

In [3]:
words = corpus.split()
vocab = sorted(set(words))
word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {val: key for key, val in word_to_index.items()}
vocab_size = len(vocab)
vocab_size

96

In [4]:
index_to_word

{0: 'a',
 1: 'after',
 2: 'ant',
 3: 'around',
 4: 'at',
 5: 'ate',
 6: 'basked',
 7: 'bear',
 8: 'bee',
 9: 'bird',
 10: 'branch',
 11: 'bush',
 12: 'butterfly',
 13: 'buzzed',
 14: 'cat',
 15: 'chased',
 16: 'chirped',
 17: 'climbed',
 18: 'crawled',
 19: 'darted',
 20: 'deer',
 21: 'dog',
 22: 'dragonfly',
 23: 'duck',
 24: 'field',
 25: 'flew',
 26: 'flower',
 27: 'fluttered',
 28: 'forest',
 29: 'fox',
 30: 'frog',
 31: 'from',
 32: 'goose',
 33: 'grass',
 34: 'grasshopper',
 35: 'grazed',
 36: 'hawk',
 37: 'hive',
 38: 'honked',
 39: 'hopped',
 40: 'howled',
 41: 'in',
 42: 'into',
 43: 'its',
 44: 'lake',
 45: 'leaped',
 46: 'left',
 47: 'light',
 48: 'line',
 49: 'lizard',
 50: 'log',
 51: 'loudly',
 52: 'lumbered',
 53: 'marched',
 54: 'mat',
 55: 'meadow',
 56: 'moon',
 57: 'moth',
 58: 'mouse',
 59: 'near',
 60: 'on',
 61: 'over',
 62: 'owl',
 63: 'perched',
 64: 'pond',
 65: 'rabbit',
 66: 'ran',
 67: 'rat',
 68: 'rested',
 69: 'rock',
 70: 'sat',
 71: 'scurried',
 72: 'sky

In [5]:
embed_dim  = 128
number_of_heads = 8
head_size  = embed_dim//number_of_heads
ffn_hidden = 512
epochs     = 100

In [6]:
embedding_table = torch.nn.Parameter(torch.randn(vocab_size, embed_dim))

In [7]:
# encoder-attention
enc_wq = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
enc_wk = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
enc_wv = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))

In [8]:
# hidden layers encoder
enc_ff1_w = torch.nn.Parameter(torch.randn(embed_dim, ffn_hidden))
enc_ff1_b = torch.nn.Parameter(torch.zeros(ffn_hidden))
enc_ff2_w = torch.nn.Parameter(torch.randn(ffn_hidden, embed_dim))
enc_ff2_b = torch.nn.Parameter(torch.zeros(embed_dim))

In [9]:
# Normalization layers
enc_norm1 = torch.nn.LayerNorm(embed_dim)
enc_norm2 = torch.nn.LayerNorm(embed_dim)
dec_norm1 = torch.nn.LayerNorm(embed_dim)
dec_norm2 = torch.nn.LayerNorm(embed_dim)
dec_norm3 = torch.nn.LayerNorm(embed_dim)

In [10]:
# decoder attention layer
dec_wq = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wk = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wv = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
cross_wq = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
cross_wk = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
cross_wv = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))

In [11]:
# decoder hidden layer
dec_ff1_w = torch.nn.Parameter(torch.randn(embed_dim, ffn_hidden))
dec_ff1_b = torch.nn.Parameter(torch.zeros(ffn_hidden))
dec_ff2_w = torch.nn.Parameter(torch.randn(ffn_hidden, embed_dim))
dec_ff2_b = torch.nn.Parameter(torch.zeros(embed_dim))
out_w = torch.nn.Parameter(torch.randn(embed_dim, vocab_size))
out_b = torch.nn.Parameter(torch.zeros(vocab_size))

In [12]:
all_params = [embedding_table, 
              enc_wq, enc_wk, enc_wv,
              enc_ff1_w, enc_ff1_b, enc_ff2_w, enc_ff2_b,
              dec_wq, dec_wk, dec_wv,
              cross_wq, cross_wk, cross_wv,
              dec_ff1_w, dec_ff1_b, dec_ff2_w, dec_ff2_b,
              out_w, out_b,
              *enc_norm1.parameters(), *enc_norm2.parameters(),
              *dec_norm1.parameters(), *dec_norm2.parameters(), *dec_norm3.parameters()
             ]

In [13]:
optimizer = torch.optim.Adam(all_params, lr=0.01)
loss_fn   = torch.nn.CrossEntropyLoss()

In [14]:
training_pairs = []
for line in corpus.strip().split('\n'):
    print(line)
    w = line.split()
    encoder_input = torch.tensor([word_to_index[x] for x in w])
    print(encoder_input)
    decoder_input = torch.tensor([word_to_index[x] for x in w[:-1]])
    print(decoder_input)
    decoder_expected_output= torch.tensor([word_to_index[x] for x in w[1:]])
    print(decoder_expected_output)
    
    training_pairs.append((encoder_input, decoder_input, decoder_expected_output))
    #print(training_pairs)

the cat sat on the mat
tensor([85, 14, 70, 60, 85, 54])
tensor([85, 14, 70, 60, 85])
tensor([14, 70, 60, 85, 54])
the dog sat on the log
tensor([85, 21, 70, 60, 85, 50])
tensor([85, 21, 70, 60, 85])
tensor([21, 70, 60, 85, 50])
the cat ate the rat
tensor([85, 14,  5, 85, 67])
tensor([85, 14,  5, 85])
tensor([14,  5, 85, 67])
the rat ran from the cat
tensor([85, 67, 66, 31, 85, 14])
tensor([85, 67, 66, 31, 85])
tensor([67, 66, 31, 85, 14])
the dog ran after the rat
tensor([85, 21, 66,  1, 85, 67])
tensor([85, 21, 66,  1, 85])
tensor([21, 66,  1, 85, 67])
the bird flew over the tree
tensor([85,  9, 25, 61, 85, 89])
tensor([85,  9, 25, 61, 85])
tensor([ 9, 25, 61, 85, 89])
the squirrel climbed the tree
tensor([85, 81, 17, 85, 89])
tensor([85, 81, 17, 85])
tensor([81, 17, 85, 89])
the rabbit hopped in the field
tensor([85, 65, 39, 41, 85, 24])
tensor([85, 65, 39, 41, 85])
tensor([65, 39, 41, 85, 24])
the fox chased the rabbit
tensor([85, 29, 15, 85, 65])
tensor([85, 29, 15, 85])
tensor([29

In [15]:
print(training_pairs)

[(tensor([85, 14, 70, 60, 85, 54]), tensor([85, 14, 70, 60, 85]), tensor([14, 70, 60, 85, 54])), (tensor([85, 21, 70, 60, 85, 50]), tensor([85, 21, 70, 60, 85]), tensor([21, 70, 60, 85, 50])), (tensor([85, 14,  5, 85, 67]), tensor([85, 14,  5, 85]), tensor([14,  5, 85, 67])), (tensor([85, 67, 66, 31, 85, 14]), tensor([85, 67, 66, 31, 85]), tensor([67, 66, 31, 85, 14])), (tensor([85, 21, 66,  1, 85, 67]), tensor([85, 21, 66,  1, 85]), tensor([21, 66,  1, 85, 67])), (tensor([85,  9, 25, 61, 85, 89]), tensor([85,  9, 25, 61, 85]), tensor([ 9, 25, 61, 85, 89])), (tensor([85, 81, 17, 85, 89]), tensor([85, 81, 17, 85]), tensor([81, 17, 85, 89])), (tensor([85, 65, 39, 41, 85, 24]), tensor([85, 65, 39, 41, 85]), tensor([65, 39, 41, 85, 24])), (tensor([85, 29, 15, 85, 65]), tensor([85, 29, 15, 85]), tensor([29, 15, 85, 65])), (tensor([85, 62, 63, 60, 85, 10]), tensor([85, 62, 63, 60, 85]), tensor([62, 63, 60, 85, 10])), (tensor([85, 58, 71, 91, 85, 11]), tensor([85, 58, 71, 91, 85]), tensor([58

In [16]:
len(encoder_input)

6

In [17]:
def pos_enc(seq_len):
    pe = torch.zeros(seq_len, embed_dim)
    for pos in range(seq_len):
        for i in range(0, embed_dim, 2):
            pe[pos, i]   = math.sin(pos / 10000 ** (i / embed_dim))
            pe[pos, i+1] = math.cos(pos / 10000 ** (i / embed_dim))
    return pe

In [18]:
if os.path.exists('transformer_weights.pth'):
    checkpoint = torch.load('transformer_weights.pth')
    
    embedding_table = checkpoint['embedding_table']
    enc_wq = checkpoint['enc_wq']
    enc_wk = checkpoint['enc_wk']
    enc_wv = checkpoint['enc_wv']
    enc_ff1_w = checkpoint['enc_ff1_w']
    enc_ff1_b = checkpoint['enc_ff1_b']
    enc_ff2_w = checkpoint['enc_ff2_w']
    enc_ff2_b = checkpoint['enc_ff2_b']
    dec_wq = checkpoint['dec_wq']
    dec_wk = checkpoint['dec_wk']
    dec_wv = checkpoint['dec_wv']
    cross_wq = checkpoint['cross_wq']
    cross_wk = checkpoint['cross_wk']
    cross_wv = checkpoint['cross_wv']
    dec_ff1_w = checkpoint['dec_ff1_w']
    dec_ff1_b = checkpoint['dec_ff1_b']
    dec_ff2_w = checkpoint['dec_ff2_w']
    dec_ff2_b = checkpoint['dec_ff2_b']
    out_w = checkpoint['out_w']
    out_b = checkpoint['out_b']
    enc_norm1.load_state_dict(checkpoint['enc_norm1'])
    enc_norm2.load_state_dict(checkpoint['enc_norm2'])
    dec_norm1.load_state_dict(checkpoint['dec_norm1'])
    dec_norm2.load_state_dict(checkpoint['dec_norm2'])
    dec_norm3.load_state_dict(checkpoint['dec_norm3'])
    
    print('Weights loaded — continuing training.')
else:
    print('No saved weights — starting fresh.')

Weights loaded — continuing training.


In [19]:
for epoch in range(1, epochs + 1):
    total_loss = 0.0

    for encoder_input, decoder_input, decoder_expected_output in training_pairs:

        encoder_input_embedding = embedding_table[encoder_input] + pos_enc(len(encoder_input))
        decoder_input_embedding = embedding_table[decoder_input] + pos_enc(len(decoder_input))
        encoder_input_shape = encoder_input_embedding.shape[0]


        Qe = encoder_input_embedding @ enc_wq
        Ke = encoder_input_embedding @ enc_wk
        Ve = encoder_input_embedding @ enc_wv

        Qe = Qe.view(encoder_input_shape, number_of_heads, head_size).transpose(0,1)
        Ke = Ke.view(encoder_input_shape, number_of_heads, head_size).transpose(0,1)
        Ve = Ve.view(encoder_input_shape, number_of_heads, head_size).transpose(0,1)
        
        scores = Qe @ Ke.transpose(-2, -1) / math.sqrt(head_size)
        attn   = torch.softmax(scores, dim=-1)
        encoder_out1 = attn @ Ve

        encoder_out1 = encoder_out1.transpose(0,1).contiguous().view(encoder_input_shape, embed_dim)
        encoder_out1 = encoder_out1 + encoder_input_embedding                      

        # Normalization layer1
        encoder_out1 = enc_norm1(encoder_out1)

        # Hidden Layers encoder
        linear_layer1 = encoder_out1 @ enc_ff1_w + enc_ff1_b
        relu1 = torch.relu(linear_layer1)
        linear_layer2 = relu1 @ enc_ff2_w + enc_ff2_b

        # Residual Layer 2 encoder
        encoder_out2 = linear_layer2 + encoder_out1 
        
        # Normalization layer 2 encoder
        encoder_out2 = enc_norm2(encoder_out2)
        
        
        # decoder masked self attention
        decoder_input_shape = decoder_input.shape[0]
        mask = torch.triu(torch.ones(decoder_input_shape, decoder_input_shape), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))

        Qd = decoder_input_embedding @ dec_wq
        Kd = decoder_input_embedding @ dec_wk
        Vd = decoder_input_embedding @ dec_wv

        Qd = Qd.view(decoder_input_shape, number_of_heads, head_size).transpose(0,1)
        Kd = Kd.view(decoder_input_shape, number_of_heads, head_size).transpose(0,1)
        Vd = Vd.view(decoder_input_shape, number_of_heads, head_size).transpose(0,1)
        
        scores_self_decoder = Qd @ Kd.transpose(-2, -1) / math.sqrt(head_size) + mask
        self_attn_decoder   = torch.softmax(scores_self_decoder, dim=-1)
        self_dec_out = self_attn_decoder @ Vd
        self_dec_out = self_dec_out.transpose(0,1).contiguous().view(decoder_input_shape, embed_dim)
        decoder_out1 = self_dec_out + decoder_input_embedding
        
        # Normalization Decoder layer 1
        decoder_out1 = dec_norm1(decoder_out1)
        
        # decoder cross attention
        # Q comes from decoder output after attention and normalization
        # K and V for decoder cross attention used output from  encoder final output
        Qcross_decoder = decoder_out1  @ cross_wq
        Kcross_decoder = encoder_out2  @ cross_wk
        Vcross_decoder = encoder_out2  @ cross_wv

        Qcross_decoder = Qcross_decoder.view(decoder_out1.shape[0], number_of_heads, head_size).transpose(0,1)
        Kcross_decoder = Kcross_decoder.view(encoder_input_shape, number_of_heads, head_size).transpose(0,1)
        Vcross_decoder = Vcross_decoder.view(encoder_input_shape, number_of_heads, head_size).transpose(0,1)
        
        scores_cross_decoder = Qcross_decoder @ Kcross_decoder.transpose(-2, -1) / math.sqrt(head_size)
        cross_attn_decoder   = torch.softmax(scores_cross_decoder, dim=-1)
        cross_out_decoder = cross_attn_decoder @ Vcross_decoder
        cross_out_decoder = cross_out_decoder.transpose(0,1).contiguous().view(decoder_out1.shape[0], embed_dim)
        decoder_out2 = decoder_out1 + cross_out_decoder
        
        # Normalization Decoder Layer 2
        decoder_out2 = dec_norm2(decoder_out2)

        decoder_linear_layer1 = decoder_out2 @ dec_ff1_w + dec_ff1_b
        decoder_relu1 = torch.relu(decoder_linear_layer1)
        decoder_linear_layer2 = decoder_relu1 @ dec_ff2_w + dec_ff2_b

        decoder_out3 = decoder_linear_layer2 + decoder_out2   
        decoder_out3 = dec_norm3(decoder_out3)
        
        # Logits Layer
        logits = decoder_out3 @ out_w + out_b                    

        # loss + backward
        loss = loss_fn(logits, decoder_expected_output)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f'Epoch {epoch:4d}   Loss: {total_loss / len(training_pairs):.4f}')

Epoch   10   Loss: 0.4126
Epoch   20   Loss: 0.3678
Epoch   30   Loss: 0.3860
Epoch   40   Loss: 0.2314
Epoch   50   Loss: 0.2317
Epoch   60   Loss: 0.1431
Epoch   70   Loss: 0.1991
Epoch   80   Loss: 0.3851
Epoch   90   Loss: 0.1570
Epoch  100   Loss: 0.0859


In [20]:
mask = torch.triu(torch.ones(5, 5), diagonal=1)
mask
mask = mask.masked_fill(mask == 1, float('-inf'))
mask

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

In [21]:
torch.save({
    'embedding_table': embedding_table,
    'enc_wq': enc_wq, 'enc_wk': enc_wk, 'enc_wv': enc_wv,
    'enc_ff1_w': enc_ff1_w, 'enc_ff1_b': enc_ff1_b,
    'enc_ff2_w': enc_ff2_w, 'enc_ff2_b': enc_ff2_b,
    'dec_wq': dec_wq, 'dec_wk': dec_wk, 'dec_wv': dec_wv,
    'cross_wq': cross_wq, 'cross_wk': cross_wk, 'cross_wv': cross_wv,
    'dec_ff1_w': dec_ff1_w, 'dec_ff1_b': dec_ff1_b,
    'dec_ff2_w': dec_ff2_w, 'dec_ff2_b': dec_ff2_b,
    'out_w': out_w, 'out_b': out_b,
    'enc_norm1': enc_norm1.state_dict(),
    'enc_norm2': enc_norm2.state_dict(),
    'dec_norm1': dec_norm1.state_dict(),
    'dec_norm2': dec_norm2.state_dict(),
    'dec_norm3': dec_norm3.state_dict(),
    'word_to_index': word_to_index,
    'index_to_word': index_to_word,
    'vocab_size': vocab_size,
    
}, 'transformer_weights.pth')

print('Weights saved.')

Weights saved.


In [22]:
128/16

8.0